# Chapter 11 Lab — Cycle Analysis

**Purpose:** replicate and understand the mechanics of the main Chapter 11 ideas (Parts 1–3: cycle basics, uncovering the cycle, Fourier).

Parts 4–5 (MESA, Hilbert, Fisher, CCI, Hurst phasing) are tested on real NIFTY data in [`03_nifty_cycle_tools/`](../../03_nifty_cycle_tools/).

This notebook is deliberately **not** an optimisation exercise and **not** a production backtest.
We use deterministic synthetic data with **known** cycles, so we can check whether each method recovers what we planted — and see how it fails.

## Learning objectives

By the end of this notebook you should be able to:
1. build a series from trend + compound cycles + noise;
2. detrend it, and see how detrending choices change the result;
3. estimate a cycle's period, amplitude and phase with trigonometric regression;
4. compute and read a periodogram (FFT);
5. demonstrate spectral leakage and the effect of a taper window;
6. test whether a spectral peak is distinguishable from noise;
7. describe when each idea can fail.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, NullFormatter

def plain_log_x(ax):
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.set_minor_formatter(NullFormatter())

rng = np.random.default_rng(11)

N = 240
t = np.arange(N)

TRUE = {'T1': 20, 'A1': 6.0, 'phi1': 0.0,
        'T2': 8,  'A2': 3.0, 'phi2': 0.0}

trend = 100 + 0.30 * t
cycle1 = TRUE['A1'] * np.sin(2 * np.pi * t / TRUE['T1'] + TRUE['phi1'])
cycle2 = TRUE['A2'] * np.sin(2 * np.pi * t / TRUE['T2'] + TRUE['phi2'])
noise = rng.normal(0, 2.0, N)

price = trend + cycle1 + cycle2 + noise
idx = pd.bdate_range('2025-01-01', periods=N)
df = pd.DataFrame({'Close': price, 'Trend': trend, 'Cycle1': cycle1, 'Cycle2': cycle2}, index=idx)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(df.index, df['Close'], label='Synthetic close')
axes[0].plot(df.index, df['Trend'], label='True trend', linestyle='--')
axes[0].set_title('Price = trend + 20-day cycle + 8-day cycle + noise')
axes[0].legend()
axes[1].plot(df.index, df['Cycle1'], label='20-day cycle (A=6)')
axes[1].plot(df.index, df['Cycle2'], label='8-day cycle (A=3)')
axes[1].plot(df.index, df['Cycle1'] + df['Cycle2'], label='Combined', linewidth=2)
axes[1].axhline(0, linewidth=1)
axes[1].set_title('Hidden components: reinforcement and cancellation')
axes[1].legend()
plt.tight_layout()
plt.show()

## 1. Detrending

`R_t = P_t - T̂_t`

Three ways to remove trend, each with a different cost:
- **Full-sample linear fit** — clean, but uses future data (look-ahead bias in a backtest).
- **Kaufman's two-EMA difference** (p. 495): longer EMA = half the dominant cycle, shorter EMA = half the longer one. For a 20-day cycle: `EMA(5) - EMA(10)`. Causal, but adds lag and distorts amplitude.
- **Triangular MA** (p. 495): centre weight 2.0, falling linearly to the ends (P = 10 → 0.4, 0.8, … 2.0 … 0.8, 0.4). Smoother, but also lagged.

The detrending method is part of the model specification.

In [ ]:
# (a) Full-sample linear trend — fine for retrospective study, NOT for backtests
beta, alpha = np.polyfit(t, df['Close'], 1)
df['Resid_linear'] = df['Close'] - (alpha + beta * t)

# (b) EMA difference (causal)
df['EMA_diff'] = df['Close'].ewm(span=5, adjust=False).mean() - df['Close'].ewm(span=10, adjust=False).mean()   # Kaufman: 20-day cycle -> 10 and 5

# (c) Kaufman's triangular average, P = 10: weights 0.4 ... 2.0 ... 0.4 (9 prices, oldest dropped)
w = np.array([0.4, 0.8, 1.2, 1.6, 2.0, 1.6, 1.2, 0.8, 0.4])
df['TMA10'] = df['Close'].rolling(9).apply(lambda x: np.dot(x, w) / w.sum(), raw=True)

true_cycle = df['Cycle1'] + df['Cycle2']
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, true_cycle, label='True cycles', linewidth=2)
ax.plot(df.index, df['Resid_linear'], label='Linear-detrended residual', alpha=0.8)
ax.plot(df.index, df['EMA_diff'], label='EMA(5) - EMA(10) (Kaufman rule)', alpha=0.8)
ax.axhline(0, linewidth=1)
ax.set_title('Different detrending methods recover different-looking cycles')
ax.legend()
plt.show()

print(f'Fitted slope {beta:.3f} (true 0.300)')
print('Correlation with true cycles:')
print('  linear residual :', round(np.corrcoef(true_cycle, df['Resid_linear'])[0, 1], 3))
mask = df['EMA_diff'].notna()
print('  EMA difference  :', round(np.corrcoef(true_cycle[mask], df['EMA_diff'][mask])[0, 1], 3))

## 2. Trigonometric regression: searching for the period

*Teaching version.* Kaufman finds the frequency directly from a factor α (`cos ω − α/2 = 0`; copper → ω = 42.9°, T = 8.4 quarters). Searching candidate periods by lowest error is an equivalent way to see the idea, not his method.

For each candidate period `T`, set `ω = 2π/T` and fit

`R_t = a·cos(ωt) + b·sin(ωt) + ε_t`

by least squares. Then `A = √(a² + b²)` and `φ = atan2(a, b)`.

The candidate with the lowest SSE is the best **in-sample** fit — not proof of a real cycle.

In [ ]:
def fit_cycle(resid, T):
    tt = np.arange(len(resid))
    w = 2 * np.pi / T
    X = np.column_stack([np.cos(w * tt), np.sin(w * tt)])
    (a, b), *_ = np.linalg.lstsq(X, resid, rcond=None)
    fitted = X @ np.array([a, b])
    sse = float(np.sum((resid - fitted) ** 2))
    return {'T': T, 'a': a, 'b': b, 'A': float(np.hypot(a, b)), 'phi': float(np.arctan2(a, b)), 'SSE': sse}

R = df['Resid_linear'].to_numpy()
periods = np.arange(4, 61, 0.5)
search = pd.DataFrame([fit_cycle(R, T) for T in periods])
best = search.loc[search['SSE'].idxmin()]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(search['T'], search['SSE'])
ax.axvline(best['T'], linestyle='--', label=f"Best T = {best['T']}")
ax.set_xlabel('Candidate period (days)')
ax.set_ylabel('SSE')
ax.set_title('Single-frequency search: lower SSE = better in-sample fit')
ax.legend()
plt.show()

print(f"Best period {best['T']:.1f} (true {TRUE['T1']}), amplitude {best['A']:.2f} (true {TRUE['A1']}), phase {best['phi']:.2f} rad (true {TRUE['phi1']})")

### Two frequencies

Remove the best cycle, search the residual again, then fit both together. More parameters fit better in-sample — that is not the same as forecasting better.

In [ ]:
c1 = fit_cycle(R, best['T'])
w1 = 2 * np.pi / c1['T']
R2 = R - (c1['a'] * np.cos(w1 * t) + c1['b'] * np.sin(w1 * t))
search2 = pd.DataFrame([fit_cycle(R2, T) for T in periods])
second = search2.loc[search2['SSE'].idxmin()]

w2 = 2 * np.pi / second['T']
X2 = np.column_stack([np.cos(w1 * t), np.sin(w1 * t), np.cos(w2 * t), np.sin(w2 * t)])
coef, *_ = np.linalg.lstsq(X2, R, rcond=None)
fit2 = X2 @ coef

print(f"Second period {second['T']:.1f} (true {TRUE['T2']}), amplitude {second['A']:.2f} (true {TRUE['A2']})")
print(f"SSE one frequency : {c1['SSE']:.0f}")
print(f"SSE two frequencies: {np.sum((R - fit2) ** 2):.0f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, R, label='Detrended residual', alpha=0.6)
ax.plot(df.index, fit2, label='Two-frequency fit', linewidth=2)
ax.axhline(0, linewidth=1)
ax.set_title('Two-frequency trigonometric fit (in-sample)')
ax.legend()
plt.show()

## 3. Fourier analysis: the periodogram

The FFT computes `a_k, b_k` for every frequency `ω_k = 2πk/N` at once.
Testable periods are **fixed by sample length**: `T_k = N/k`. Power ∝ amplitude².

In [ ]:
def periodogram(x, taper=False):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    if taper:
        x = x * np.hanning(len(x))
    F = np.fft.rfft(x)
    k = np.arange(len(F))
    power = np.abs(F) ** 2 / len(x)
    out = pd.DataFrame({'k': k, 'period': np.where(k > 0, len(x) / np.maximum(k, 1), np.inf), 'power': power})
    return out[out['k'] >= 2]   # drop the mean and the single longest period

pg = periodogram(R)
pure_noise = periodogram(rng.normal(0, 2.0, N))

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(pg['period'], pg['power'], marker='o', markersize=3, label='Detrended price')
ax.semilogy(pure_noise['period'], pure_noise['power'], alpha=0.6, label='Pure noise, same size')
plain_log_x(ax)
ax.set_xlabel('Period (days, log scale)')
ax.set_ylabel('Power (log scale)')
ax.set_title('Periodogram: spikes where the hidden cycles are')
ax.legend()
plt.show()

top = pg.nlargest(3, 'power')[['period', 'power']]
print(top.round(1).to_string(index=False))
ratio = top['power'].iloc[0] / top['power'].iloc[1]
print(f"Power ratio of top two peaks: {ratio:.1f}  (amplitude ratio² = {(TRUE['A1'] / TRUE['A2']) ** 2:.1f})")

## 4. Spectral leakage

With `N = 120`, the testable periods near 20 are 24, 20, 17.1 … A true **22-day** cycle has no exact slot, so its power spreads into neighbouring bins.
A **Hann taper** fades the sample's edges, reducing the false power that the wrap-around creates far from the peak.

Notice the trade-off in the left panel: the taper **widens** a peak that was already exact. Tapering buys less distant leakage at the cost of frequency resolution.

In [ ]:
n_short = 120
ts = np.arange(n_short)
exact = 6 * np.sin(2 * np.pi * ts / 20)
off_grid = 6 * np.sin(2 * np.pi * ts / 22)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, series, title in [(axes[0], exact, 'True period 20 (on the grid)'),
                          (axes[1], off_grid, 'True period 22 (between grid points)')]:
    raw = periodogram(series)
    hann = periodogram(series, taper=True)
    ax.semilogy(raw['period'], raw['power'] + 1e-6, marker='o', markersize=3, label='No taper')
    ax.semilogy(hann['period'], hann['power'] + 1e-6, marker='x', markersize=3, label='Hann taper')
    plain_log_x(ax)
    ax.set_title(title)
    ax.set_xlabel('Period (days)')
    ax.legend()
axes[0].set_ylabel('Power')
plt.tight_layout()
plt.show()

raw22 = periodogram(off_grid)
share = raw22['power'].max() / raw22['power'].sum()
print(f"Off-grid cycle: largest bin holds only {share:.0%} of total power (the rest has leaked)")

## 5. Is the peak real? A permutation check

Shuffling the residuals destroys any time structure but keeps the same values.
If the largest peak in our data is bigger than the largest peak in (almost) every shuffled series, it is unlikely to be a noise artefact.

Caveat: shuffling assumes independent observations. Real prices have **red noise** (more power at long periods), so this test is too generous for long periods.

In [ ]:
observed_max = pg['power'].max()
n_perm = 1000
perm_rng = np.random.default_rng(2026)
null_max = np.array([periodogram(perm_rng.permutation(R))['power'].max() for _ in range(n_perm)])
threshold = np.quantile(null_max, 0.95)
p_value = (1 + np.sum(null_max >= observed_max)) / (1 + n_perm)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(null_max, bins=40, alpha=0.7, label='Largest peak in shuffled series')
ax.axvline(threshold, linestyle='--', label='95% noise threshold')
ax.axvline(observed_max, linewidth=2, label='Largest peak in our data')
ax.set_xlabel('Maximum periodogram power')
ax.set_title('Permutation test on the maximum peak (accounts for searching all bins)')
ax.legend()
plt.show()

print(f'Observed max {observed_max:.0f}, 95% noise threshold {threshold:.0f}, p = {p_value:.4f}')

### Experiment: make the cycle disappear

Change `A1` to `1.0` and `A2` to `0.5` in the first cell and rerun.
At what amplitude does the 20-day peak stop beating the noise threshold? That is the practical meaning of *amplitude relative to noise*.

In [ ]:
assert abs(best['T'] - TRUE['T1']) <= 1, 'regression should recover the 20-day cycle'
assert abs(second['T'] - TRUE['T2']) <= 0.5, 'second pass should recover the 8-day cycle'
assert abs(top['period'].iloc[0] - TRUE['T1']) < 1e-9, 'FFT top peak should be at 20 days'
assert observed_max > threshold, 'planted cycle should beat the noise threshold'
assert np.isclose(np.sum(np.abs(np.fft.rfft(exact)) ** 2 > 1e-6), 1), '20-day cycle sits in exactly one bin when N=120'
print('Basic cycle-analysis sanity checks passed.')

## 6. What is missing before this becomes a strategy?

- causal (rolling / expanding) detrending and cycle estimation — no look-ahead;
- a phase estimate at decision time, and a rule that turns phase into entries/exits (Parts 4–5 — tested on NIFTY in `03_nifty_cycle_tools/`: no edge without look-ahead);
- handling of cycles whose period drifts;
- exit rule, stop / invalidation rule, position sizing;
- transaction costs and slippage;
- out-of-sample validation and robustness across instruments.

That is why this notebook stops here.

## Reflection

1. Why did the full-sample linear detrend fit well here, and why is it unusable in a backtest?
2. Why does the EMA difference recover the cycle with lag and changed amplitude?
3. Why is the lowest-SSE period only an in-sample candidate?
4. What decides which periods an FFT can test?
5. What happened to the 22-day cycle's power, and what did the taper change?
6. Why does the permutation test use the **maximum** peak rather than the peak at 20 days?
7. Why is the permutation test too generous for long periods on real price data?